In [47]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float", lambda x: f"{x:,.2f}")

DB_PATH = r"../customer_data/vault_chain_flow.db"
TABLE = "customer_Transactions"

In [48]:
con = sqlite3.connect(DB_PATH)
cursor = con.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables in database:", cursor.fetchall())

Tables in database: [('Customer_Transactions',)]


In [49]:
df = pd.read_sql_query(f"SELECT * FROM Customer_Transactions", con)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns.")

Loaded 580,962 rows and 14 columns.


In [50]:
print("shape (rows, columns):", df.shape)
print("Total rows:", f"{df.shape[0]:,}")
print("Total columns:", df.shape[1])

shape (rows, columns): (580962, 14)
Total rows: 580,962
Total columns: 14


In [51]:
col_info = pd.DataFrame({
    "column": df.columns,
    "Dtype": df.dtypes.astype(str).values,
    "Non-Null": df.notna().sum().values,
    "Null": df.isna().sum().values,
    "Null %": (df.isna().mean() * 100).round(2).values,
    "unique": df.nunique(dropna=True).values,
})
print(col_info.to_string(index=False))

             column   Dtype  Non-Null  Null  Null %  unique
      TransactionID     str    580962     0    0.00  580962
         CustomerID     str    580962     0    0.00    4997
    TransactionDate     str    580962     0    0.00     639
    TransactionTime     str    580962     0    0.00   71454
    TransactionType     str    580962     0    0.00       9
  TransactionAmount float64    580962     0    0.00  306886
TransactionCurrency     str    569116 11846    2.04       6
          AssetType     str    566487 14475    2.49       5
        CustomerDOB     str    580962     0    0.00    4393
         CustGender     str    580962     0    0.00       2
       CustLocation     str    563766 17196    2.96      23
 CustAccountBalance float64    580962     0    0.00  407472
     WalletActivity   int64    580962     0    0.00       2
  TransactionStatus     str    580962     0    0.00       5


Missing Values Per Column

Quick one-liner that returns the total count of NAN values fro every column in df
Useful as a fast inline check of data completeness especially right after parsing dates (eg, pd.to_datetime silently convert unparseable values to Nat, which shows up here as missing)

Compare against the earlier missing DataFrame to confirm nothing new was introduced by recent transactions



In [52]:
df.isna().sum()

TransactionID              0
CustomerID                 0
TransactionDate            0
TransactionTime            0
TransactionType            0
TransactionAmount          0
TransactionCurrency    11846
AssetType              14475
CustomerDOB                0
CustGender                 0
CustLocation           17196
CustAccountBalance         0
WalletActivity             0
TransactionStatus          0
dtype: int64

Imputing Missing TransactionCurrency with a Rule-Based Cascade

TransactionCurrency is a critical field - every transaction should have a currency for aggregation. FX conversion and reconcillation. Rather than dropping rows or applying a naive global fill (eg, model) we apply a prioritized cascade of 5 rules,  ordered from most-specific and reliable to most-generic fallback. Each rule targets only the rows still missing a currency. so earlier, higher-confidence rules always take precedence.

A shared lookup table maps each ASSetType to it natural settlement currency: BTC->BTC, ETH-->ETH, USDT--> USDT, and stablecoin --> USDT (since stablecoins settle in USDT)

RULE 1 AssetType drives currency

Logic: If TransactionCurrency is missing and the row has a recognized AssetType , infer the currency directly from the asset.

Why first? Asset type is the strongest signal --a BYC transaction is almost certainly denominated in BTC.
The mask ensures we only touch rows that are both missing currency and have a known asset
Print how many rows this rule filled

RULE 2: Crypto transaction type -. currency from asset

Logic: Crypto Buy/sell transactions with a known asset inherit that asset"s currency -- even if Rule 1 didnt already catch them
Why here Reinforces the crypto--> asset currency assumption using transactionType as an additional qualifier
Requires Assettype to be present so the lookup producesa valid value.

RULE 3: Wallet - only movement default to GBP
Logic: Rows flagged as wallet activity with no currency are interanl wallet movements which defaultbto the home currency (GBP)
why GBP? internal wallet activity doesnt cross borbers so it settle in the platform"s home currency 

RULE 4: 
Cross-border defaults to USD
Logic: Cross-Border payement transactions without a currency default to USD the de facto standard for international settlement

RULE 5: Everthing else --> GBP (home currency)
Logic: Catch all fallback-- any remaining rows missing a currency are filled with GBP.
Why last? This is the lowest-confidence rule and should only apply after allspecific rules have had a chance to fire. it gurantees zero remaining nulls

Final Verification
confirms the casacade fully resolved the column (expected: 0 remaining nulls)

KEY DESIGN PRINCIPLES
principle       why it matters
Ordered specificity     Rules run from most-specific (asset -> currency) to most-genericfallbacl --> GBP so high
                        confidence inference always win.
Null-guarded masks       Every rule only applies where the currency is still missingso no existing values                 ever                overwrtten

observability:  each rule point how many rows it filled--you can audit which rules heavy liftting and detect unexpected shift in data ditribution

idempotent and reproducible  Rules are deterministic re-running the block on an already-filled column is ano-op

In [53]:
# Rule 1 AssetType drives currency
asset_to_currency = {
    "BTC": "BTC",
    "ETH": "ETH",
    "USDT": "USDT",
    "Stablecoin": "USDT",
}
mask = df["TransactionCurrency"].isna() & df["AssetType"].isin(asset_to_currency)
df.loc[mask, "TransactionCurrency"] = df.loc[mask, "AssetType"].map(asset_to_currency)
print(f"Rule 1 filled: {mask.sum():,}")



# Rule 2 - wallet - only movements default to GBP
mask = df["TransactionCurrency"].isna() & (df["WalletActivity"] == 1)
df.loc[mask, "TransactionCurrency"] = "GBP"
print(f"Rule 2 filled: {mask.sum():,}")



# Rule 3 - Everything else-> GBP (home currency)
mask = df["TransactionCurrency"].isna()
df.loc[mask, "TransactionCurrency"] = "GBP"
print(f"Rule 3 filled: {mask.sum():,}")

print("\nRemaining TransactionCurrency nulls:", df["TransactionCurrency"].isna().sum())

Rule 1 filled: 3,217
Rule 2 filled: 1,835
Rule 3 filled: 6,794

Remaining TransactionCurrency nulls: 0


Imputation Missing AssetType TransactionCurrency

Now that TransactionCurrency is fully populated (from the prvious cascade), we can reverse-infer AssetType for rows where it's missing. Becos currency and asset type are tightly coupled - fiat currencies map to traditional assets, stablecoins to stablecoin assets, and crypto currencies to their matching crypto asset - this is a reliable , deterministic fill.

A shared reverse-lookup table maps each currency to its corresponding asset class:

GBP,EUR,USD --> Traditional
USDT----> Stablecoin
BTC ---> BTC
ETH  ---> ETH

Step 1 --- Fill AssetType from TransactionCurrency
Logic: for every row where AssetType is missing, look up the asset class based on the row"s TransactionCurrency and assign it.

whyy this works : The previous step guaranteed every rows has a currency, so the lookup can resolve almost ever remaining null

A Mask catpure rows where AssetType is null before the fill, so we can report exactly how many rows were updated.
prints the count of rows filled

Step 2--Report remaining Nulls
After the fill, print how many AssetType values are still missing. In theory this should be0, since every currency in the lookup map to an asset-- but it"s good practice to verify rather than assume.

Step 3 ---safety net fallback

Any row still missing AssetType (eg, if a currency existed that wasn't in the lookup table) is filled with "Traditional" using a simple fillna.

why "Traditional" ? Its the most common asset class in the dataset and a conservative default -- testing unknownrows as traditional fiat--backed activity rather than crypto,

Why this steps exists: Defensive programming--it guarantees the column has zero nulls downstream, so no later analysis or model breaks on missing values.

In [54]:
currency_to_asset = {
    "GBP": "Traditional",
    "EUR": "Traditional",
    "USD": "Traditional",
    "USDT": "Stablecoin",
    "BTC": "BTC",
    "ETH": "ETH",
}

mask = df["AssetType"].isna()
df.loc[mask, "AssetType"] = df.loc[mask, "TransactionCurrency"].map(currency_to_asset)

print(f"AssetType filled from currency: {mask.sum():,}")
print("remaining AssetType nulls:", df["AssetType"].isna().sum())

# safety net  anything still null (shouldnt be) --Traditional
df["AssetType"] = df["AssetType"].fillna("Traditional")

AssetType filled from currency: 14,475
remaining AssetType nulls: 0


Imputing Missing CustLocation via Customer Mode + Fallback

Customer location is often missing on individual transactions but can be reliably inferred from the customer's other transactions--most customers operate from a consistent location, This step applies a two-tier imputation:

Step 1  Compute each customer"s model location
For every CustomerID, Calculate the most frequency occuring non-null CustLocation across all of that customer"s transactions

uses groupby("CustomerID").transform(...) so the result is a Series aligned to the original DataFrame index --nomerge or temp column needed.
The lambda drops null first, so missing values dont skew the mode.
If a Customer has no recorded location at all the result is NAN (handled in step 2)

Step 2---Fill nulls with the Customer's Mode then fall back to "unknown"

Two chained fillna calls resolve the column in priority order:
1 Fill from the customer's model location --high -confidence inference for customers who have at least one knownlocation elsewhere
2   Fall back to "unknown"--- for customers with zero known locations, we assign a sentinel value so the column has no null downstream.

The chained pattern reads top-to-bottom as: "use the mode if we have one, otherwise mark it unknown"

Step 3---Verify

print the remaining null count in CustLocation--expected to be 0, confirming the column is fully populated.

In [57]:
mode_per_customer = df.groupby("CustomerID")["CustLocation"].transform(
    lambda s: s.dropna().mode().iloc[0] if s.notna().any() else np.nan
)

df["CustLocation"] = df["CustLocation"].fillna(mode_per_customer).fillna("unknown")

print("Remaining CustLocation nulls:", df["CustLocation"].isna().sum())

Remaining CustLocation nulls: 0


In [59]:
df["CustLocation"].value_counts()

CustLocation
London                  122409
Manchester               46714
Birmingham               43160
Leeds                    35836
Glasgow                  30496
Bristol                  29452
Liverpool                28112
Edinburgh                25299
Sheffield                21724
Belfast                  20468
Newcastle                18798
Leicester                18361
Cardiff                  17852
Brighton                 17841
Southampton              17704
Nottingham               16845
International - EU       12046
Milton Keynes            11890
Coventry                 11178
Reading                  10281
Aberdeen                  9601
International - US        7955
International - APAC      6936
unknown                      4
Name: count, dtype: int64

In [60]:
df.isna().sum()

TransactionID          0
CustomerID             0
TransactionDate        0
TransactionTime        0
TransactionType        0
TransactionAmount      0
TransactionCurrency    0
AssetType              0
CustomerDOB            0
CustGender             0
CustLocation           0
CustAccountBalance     0
WalletActivity         0
TransactionStatus      0
dtype: int64

In [61]:
df.head(200)

,TransactionID,CustomerID,TransactionDate,TransactionTime,TransactionType,TransactionAmount,TransactionCurrency,AssetType,CustomerDOB,CustGender,CustLocation,CustAccountBalance,WalletActivity,TransactionStatus
0,TXN00000001,CUST00001,2024-12-13,10:03:23,P2P Transfer,79.36,GBP,Traditional,2007-04-30,Male,London,"4,716.01",0,Completed
1,TXN00000002,CUST00001,2024-12-13,17:15:53,Bill Payment,54.95,GBP,Traditional,2007-04-30,Male,London,"4,661.06",0,Completed
2,TXN00000003,CUST00001,2024-12-14,12:26:16,Wallet Transfer,410.02,GBP,Traditional,2007-04-30,Male,London,"5,071.08",1,Completed
3,TXN00000004,CUST00001,2024-12-14,13:10:09,Withdrawal,212.52,USD,Traditional,2007-04-30,Male,London,"4,905.31",0,Completed
4,TXN00000005,CUST00001,2024-12-14,20:13:04,Bill Payment,674.77,GBP,Traditional,2007-04-30,Male,London,"4,230.54",0,Completed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,TXN00000196,CUST00002,2025-02-16,13:46:54,P2P Transfer,"1,581.49",GBP,Traditional,1961-08-07,Male,Newcastle,"5,332.95",0,Completed
196,TXN00000197,CUST00002,2025-02-24,12:24:48,P2P Transfer,625.62,GBP,Traditional,1961-08-07,Male,Newcastle,"5,958.56",0,Completed
197,TXN00000198,CUST00002,2025-03-03,09:00:13,Card Payment,"1,068.54",GBP,Traditional,1961-08-07,Male,Newcastle,"4,890.02",0,Completed
198,TXN00000199,CUST00002,2025-03-05,08:25:25,P2P Transfer,800.54,USD,Traditional,1961-08-07,Male,Newcastle,"4,265.60",0,Completed


In [63]:
df.duplicated().sum()
df["TransactionID"].duplicated().sum()

np.int64(0)

Now create artifacts folder inside customer_data

In [64]:
import os
os.makedirs("../customer_data/artifacts/", exist_ok=True)

In [65]:
df.to_csv("../customer_data/artifacts/cleaned_transaction_data.csv", index=False)